In [37]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy import stats

pd.set_option('display.max_columns', None)

In [38]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .master("local[*]")
    .appName("threat_score_impact_analysis")
    .getOrCreate()
    )

In [39]:
# caminho pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"

## Validação do Impacto na Ameaça (threat_score_impact)

In [40]:
# base de ameaça criada
threat_dataset_path = str(data_folder_path / "threat_dataset")
df_threat = spark.read.parquet(threat_dataset_path)

In [ ]:
# Referencia: build_aggregated_df tambem existe em src/utils/spark_aggregation.py;
# join_match_stats, build_side_df e build_defending_side_df tambem existem em
# src/utils/team_perspective.py. Ainda nao importados/usados daqui.
def build_aggregated_df(
    df,
    group_cols,
    agg_cols,
    agg_func,
    agg_prefix
):
    """
    Agrega um DataFrame utilizando a função de agregação desejada.
    """

    # expressões de agregação com o prefixo conforme a agregação realizada (ex: média -> avg, máximo -> max)
    agg_exprs = [
        F.round(agg_func(F.col(c)), 3).alias(f"{agg_prefix}_{c}")
        for c in agg_cols
    ]

    # DataFrame agregado
    df_agg = (
        df
        .groupBy(*group_cols)
        .agg(*agg_exprs)
    )

    # nomes das colunas agregadas
    agg_cols_result = [
        f"{agg_prefix}_{c}"
        for c in agg_cols
    ]

    return agg_cols_result, df_agg


def join_match_stats(df, df_match_stats):

    # adiciona estatísticas da partida
    return (
        df.join(
            df_match_stats,
            on=[
                "date",
                "homeTeamName",
                "opponentTeamName"
            ],
            how="left"
        )
    )


def build_side_df(df, is_home, avg_cols):

    # filtra mandante ou visitante
    side_filter = F.col('homeTeam') if is_home else ~F.col('homeTeam')

    # mapeamento das colunas conforme o lado
    cols = {
        "teamName": "homeTeamName" if is_home else "opponentTeamName",
        "win": (F.col("FTR") == ('H' if is_home else 'A')),
        "goals": "FTHG" if is_home else "FTAG",
        "shots": "HS" if is_home else "AS",
        "shots_target": "HST" if is_home else "AST",
        "avg_win_odds": "AvgH" if is_home else "AvgA",
    }

    # colunas fixas
    fixed_select = [
        'competitionName',
        'season',
        'gameId',
        "date",
        F.col(cols["teamName"]).alias("teamName"),
    ]

    # métricas agregadas
    avg_select = [F.col(c) for c in avg_cols]

    # estatísticas da partida
    result_select = [
        cols["win"].alias("win"),
        F.col(cols["goals"]).alias("goals"),
        F.col(cols["shots"]).alias("shots"),
        F.col(cols["shots_target"]).alias("shots_target"),
        F.col(cols["avg_win_odds"]).alias("avg_win_odds"),
    ]

    return (
        df
        .filter(side_filter)
        .select(*fixed_select, *avg_select, *result_select)
    )


def build_defending_side_df(df, is_home, avg_cols):
    """
    Espelha build_side_df, mas pra medir threat_score_impact do lado que
    DEFENDEU no ciclo/evento, não do lado que atacava — e associa a esse
    time as estatísticas de partida que ele SOFREU (do adversário), não as
    que ele gerou.

    Por que a perspectiva é invertida (duas inversões, nesta função):

    1) Filtro de lado (linhas -> time): em build_side_df, a flag `homeTeam`
       indica quem tinha a POSSE (estava atacando), e a métrica agregada é
       atribuída a esse mesmo time atacante — faz sentido pro threat_score,
       que mede a ameaça GERADA por quem ataca. threat_score_impact mede o
       efeito de um evento normalmente DEFENSIVO (uma interceptação, um
       desarme) na ameaça — o time relevante pra essa métrica é quem estava
       DEFENDENDO, ou seja, o lado OPOSTO ao indicado por `homeTeam`. Por
       isso `side_filter` aqui é o inverso de build_side_df: is_home=True
       seleciona linhas onde o mandante DEFENDEU (`~homeTeam`, isto é, o
       visitante estava com a posse), não onde ele atacou.

    2) Estatísticas de partida (colunas de resultado): como a métrica agora
       representa o que o time DEFENDEU/evitou, não faz sentido correlacionar
       o impacto defensivo de um time com os gols que ELE fez — o que
       importa é o que ele SOFREU. Por isso "goals"/"shots"/"shots_target"
       apontam pras colunas do ADVERSÁRIO na mesma partida (gols sofridos =
       gols do outro lado, chutes sofridos = chutes do outro lado, etc.), e
       "avg_win_odds" usa a odds de vitória do ADVERSÁRIO — a lógica de
       "o quão favorito era quem eu enfrentei", não "o quão favorito eu era".
       "win" NÃO é espelhado: resultado da própria partida (o time em
       questão venceu ou não) é um fato sobre ele mesmo, não algo "sofrido"
       ou "gerado", então segue igual a build_side_df.
    """
    # filtro de lado INVERTIDO em relação a build_side_df: seleciona quem
    # DEFENDEU (não quem atacou) naquele ciclo/evento
    side_filter = ~F.col('homeTeam') if is_home else F.col('homeTeam')

    # mapeamento das colunas conforme o lado, mas com gols/chutes/odds
    # SOFRIDOS (do adversário na mesma partida), não gerados pelo próprio time
    cols = {
        "teamName": "homeTeamName" if is_home else "opponentTeamName",
        "win": (F.col("FTR") == ('H' if is_home else 'A')),  # resultado da própria partida: não espelha
        "goals": "FTAG" if is_home else "FTHG",              # gols sofridos = gols do adversário
        "shots": "AS" if is_home else "HS",                  # chutes sofridos = chutes do adversário
        "shots_target": "AST" if is_home else "HST",         # chutes a gol sofridos = chutes a gol do adversário
        "avg_win_odds": "AvgA" if is_home else "AvgH",       # odds relevante = odds de vitória do adversário
    }

    # colunas fixas
    fixed_select = [
        'competitionName',
        'season',
        'gameId',
        "date",
        F.col(cols["teamName"]).alias("teamName"),
    ]

    # métricas agregadas
    avg_select = [F.col(c) for c in avg_cols]

    # estatísticas da partida (já "espelhadas" pro adversário via cols acima)
    result_select = [
        cols["win"].alias("win"),
        F.col(cols["goals"]).alias("goals"),
        F.col(cols["shots"]).alias("shots"),
        F.col(cols["shots_target"]).alias("shots_target"),
        F.col(cols["avg_win_odds"]).alias("avg_win_odds"),
    ]

    return (
        df
        .filter(side_filter)
        .select(*fixed_select, *avg_select, *result_select)
    )

In [ ]:
# Referencia: tambem existe em src/utils/team_match_correlation_plots.py
# (ainda nao importado/usado daqui).
def plot_correlation_heatmap(df_agg, corr_cols):
    """
    Constrói o df agregado (via build_aggregated_df), converte para pandas,
    calcula a matriz de correlação das colunas em `corr_cols` e plota um
    heatmap com Plotly.
    """

    df_agg_pd = df_agg.toPandas()

    corr = df_agg_pd[corr_cols].corr(method='spearman')

    fig = go.Figure(
        data=go.Heatmap(
            z=corr.values,
            x=corr.columns,
            y=corr.columns,
            colorscale='RdBu', 
            zmin=-1, 
            zmax=1,
            text=corr.round(2).values,
            texttemplate="%{text}",
            colorbar=dict(title="Correlação"),
        )
    )

    fig.update_layout(
        title="Matriz de Correlação",
        template="simple_white",
        width=1500,
        height=900,
        xaxis=dict(tickangle=-45),
    )

    fig.show()

In [ ]:
# Referencia: tambem existe em src/utils/team_match_correlation_plots.py
# (ainda nao importado/usado daqui).
def plot_pvalue_heatmap(df_agg, corr_cols, method='spearman', alpha=0.05):
    """
    Calcula a matriz de p-valor (via scipy) pro mesmo conjunto de colunas
    usado no heatmap de correlação, e plota com Plotly. Célula com
    p-valor < alpha recebe um "*" ao lado do número pra marcar significância.
    Usa pares completos (dropna por par de colunas), igual ao .corr() do pandas.
    """

    df_agg_pd = df_agg.toPandas()

    corr_func = stats.spearmanr if method == 'spearman' else stats.pearsonr

    # matriz de p-valor, mesma ordem/eixos do heatmap de correlação
    pvals = pd.DataFrame(np.nan, index=corr_cols, columns=corr_cols)

    for i, col_i in enumerate(corr_cols):
        for j, col_j in enumerate(corr_cols):
            if j < i:
                continue
            if i == j:
                # correlação de uma coluna com ela mesma é trivial (r=1, p=0);
                # calcular via scipy aqui quebraria (colunas duplicadas no select)
                pvals.loc[col_i, col_j] = 0.0
                continue
            paired = df_agg_pd[[col_i, col_j]].dropna()
            pvalue = np.nan if len(paired) < 3 else corr_func(paired[col_i], paired[col_j])[1]
            pvals.loc[col_i, col_j] = pvalue
            pvals.loc[col_j, col_i] = pvalue

    # texto da célula com "*" pra marcar p-valor < alpha
    text = pvals.round(3).astype(str)
    text = text.where(pvals >= alpha, text + '*')

    fig = go.Figure(
        data=go.Heatmap(
            z=pvals.values,
            x=pvals.columns,
            y=pvals.columns,
            colorscale='Blues',
            reversescale=True,
            zmin=0,
            zmax=1,
            text=text.values,
            texttemplate="%{text}",
            colorbar=dict(title="p-valor"),
        )
    )

    fig.update_layout(
        title=f"Matriz de p-valor ({method}) — * indica p < {alpha}",
        template="simple_white",
        width=1500,
        height=900,
        xaxis=dict(tickangle=-45),
    )

    fig.show()

In [ ]:
# Referencia: plot_histogram/plot_boxplot/plot_line tambem existem em
# src/utils/exploratory_plots.py (ainda nao importados/usados daqui).
def plot_histogram(df, column, nbins=None, title=None, color="#2E5EAA"):
    pdf = df.select(column).toPandas()

    fig = go.Figure()
    fig.add_trace(
        go.Histogram(
            x=pdf[column],
            nbinsx=nbins,
            marker=dict(color=color),
            name=column,
        )
    )

    fig.update_layout(
        template="simple_white",
        title=title or f"Distribuição de {column}",
        xaxis_title=column,
        yaxis_title="Frequência",
        height=600,
        width=1000
    )

    fig.show()
    
def plot_boxplot(df, column, title=None, color="#2E5EAA"):
    pdf = df.select(column).toPandas()

    fig = go.Figure()
    fig.add_trace(
        go.Box(
            y=pdf[column],
            marker=dict(color=color),
            name=column,
            boxpoints="outliers",
        )
    )

    fig.update_layout(
        template="simple_white",
        title=title or f"Boxplot de {column}",
        yaxis_title=column,
        height=600,
        width=1000
    )

    fig.show()
    
def plot_line(df, column, x=None, title=None, color="#2E5EAA"):
    cols = [x, column] if x is not None else [column]
    pdf = df.select(*cols).toPandas()

    x_values = pdf[x] if x is not None else pdf.index
    median = float(pdf[column].median())

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=pdf[column],
            mode="lines",
            line=dict(color=color, width=2),
            name=column,
        )
    )

    fig.add_shape(
        type="line",
        xref="paper",
        x0=0,
        x1=1,
        yref="y",
        y0=median,
        y1=median,
        line=dict(color="red", dash="dash", width=2),
    )

    fig.update_layout(
        template="simple_white",
        title=title or f"{column} ao longo do tempo",
        xaxis_title=x or "Índice",
        yaxis_title=column,
        height=600,
        width=1500,
    )

    fig.show()

In [45]:
df_threat.printSchema()

root
 |-- gameId: long (nullable = true)
 |-- competitionId: long (nullable = true)
 |-- season: string (nullable = true)
 |-- eventId: string (nullable = true)
 |-- period: long (nullable = true)
 |-- periodDescription: string (nullable = true)
 |-- eventType: string (nullable = true)
 |-- eventTypeDescription: string (nullable = true)
 |-- startGameClock: long (nullable = true)
 |-- startFormattedGameClock: string (nullable = true)
 |-- homeTeam: boolean (nullable = true)
 |-- details_parsed: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- eventPlayerId: long (nullable = true)
 |-- eventPlayerName: string (nullable = true)
 |-- eventTeamId: long (nullable = true)
 |-- eventTeamName: string (nullable = true)
 |-- eventSubTypeDescription: string (nullable = true)
 |-- eventOutcomeDescription: string (nullable = true)
 |-- competitionName: string (nullable = true)
 |-- date: string (nullable = true)
 |-- venueType: string (nullable = t

In [46]:
df_possession_agg = (
    df_threat
    .groupBy('gameId', 'possession_id')
    .agg(F.countDistinct(F.col('eventId')).alias('qtd_eventos')
    )
    .sort('qtd_eventos', ascending=False)
)

print(df_possession_agg.count())
df_possession_agg.show()

100930
+------+-------------+-----------+
|gameId|possession_id|qtd_eventos|
+------+-------------+-----------+
|  4721|          178|         84|
|  4572|          154|         64|
|  4807|          198|         64|
|  4654|           28|         62|
|  4699|          126|         61|
|  4779|          213|         61|
|  4700|          287|         61|
|  4619|           71|         61|
|  4711|           71|         61|
|  4764|           73|         60|
|  4760|           84|         59|
|  4589|           91|         57|
|  4673|           24|         57|
|  4664|           20|         55|
|  4764|           45|         55|
|  4739|          194|         54|
|  4779|           69|         54|
|  4689|           85|         54|
|  4721|          164|         54|
|  4700|          293|         53|
+------+-------------+-----------+
only showing top 20 rows


In [47]:
df_possession_agg_pd = df_possession_agg.toPandas()

fig = go.Figure()

#cores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig.add_trace(
    go.Box(
        y=df_possession_agg_pd['qtd_eventos'],
        name='',
        marker_color='#1f77b4',
        text=df_possession_agg_pd['possession_id'],
        hovertemplate='%{text}<br>Quantidade de eventos: %{y}<extra></extra>',
        #legendgroup=freq,
        #showlegend=True
    )
)

fig.update_layout(
    title=f'Ciclos de posse das partidas',
    height=700,
    width=600
)

fig.update_yaxes(title_text='Quantidade de eventos')
fig.show()

In [48]:
(
    df_threat
    .groupBy('eventTypeDescription')
    .agg(
        F.round(F.min(F.col('threat_score_impact')), 3).alias('min_threat_score_impact'),
        F.round(F.avg(F.col('threat_score_impact')), 3).alias('avg_threat_score_impact'),
        F.round(F.median(F.col('threat_score_impact')), 3).alias('median_threat_score_impact'),
        F.round(F.max(F.col('threat_score_impact')), 3).alias('max_threat_score_impact'),
        )
    .sort('median_threat_score_impact', ascending=False)
).show(truncate=False)

+--------------------+-----------------------+-----------------------+--------------------------+-----------------------+
|eventTypeDescription|min_threat_score_impact|avg_threat_score_impact|median_threat_score_impact|max_threat_score_impact|
+--------------------+-----------------------+-----------------------+--------------------------+-----------------------+
|Second half kick off|-0.407                 |-0.04                  |0.0                       |0.416                  |
|Pass                |-0.838                 |-0.046                 |0.0                       |0.556                  |
|First half kick off |-0.436                 |-0.025                 |-0.007                    |0.212                  |
|Ball Carry          |-0.831                 |-0.123                 |-0.027                    |0.48                   |
|Challenge           |-0.844                 |-0.176                 |-0.062                    |0.517                  |
|Clearance           |-0

In [49]:
# # Distribuição do impacto na ameaça de eventos de todas as partidas da temporada
# plot_histogram(df_threat, 'threat_score_impact')
# plot_boxplot(df_threat, 'threat_score_impact')

# # impacto na ameaça de eventos defensivos sequenciais em uma partida específica da temporada
# game_id = 4438
# plot_line(df_threat.filter((F.col('gameId') == game_id)), 'threat_score_impact')

#### 1.2. Preparação da base de odds

In [50]:
# base de odds da Premier League 2022-2023
match_stats_22_23_path = str(data_folder_path / "match_stats" / "PL_22_23.csv")

df_pl_match_stats_22_23 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(match_stats_22_23_path , sep=',')

In [51]:
team_name_mapping = {
    "Tottenham": "Tottenham Hotspur",
    "Brighton": "Brighton & Hove Albion",
    "Man City": "Manchester City",
    "Crystal Palace": "Crystal Palace",
    "Leicester": "Leicester City",
    "Aston Villa": "Aston Villa",
    "Bournemouth": "AFC Bournemouth",
    "Fulham": "Fulham",
    "West Ham": "West Ham",
    "Man United": "Manchester United",
    "Wolves": "Wolverhampton Wanderers",
    "Southampton": "Southampton",
    "Liverpool": "Liverpool",
    "Chelsea": "Chelsea",
    "Nott'm Forest": "Nottingham Forest",
    "Newcastle": "Newcastle United",
    "Everton": "Everton",
    "Leeds": "Leeds United",
    "Arsenal": "Arsenal",
    "Brentford": "Brentford"
}

df_pl_match_stats_22_23_filtrado_mapped = (
    df_pl_match_stats_22_23
    .select(
    F.to_date(F.col("Date"), "dd/MM/yyyy").alias("date"),
    F.col('HomeTeam').alias('homeTeamName'),
    F.col('AwayTeam').alias('opponentTeamName'),
    'FTHG',
    'FTAG',
    'FTR',
    'HS',
    'AS',
    'HST',
    'AST',    
    'AvgH',
    'AvgA',
    'AvgD'
    )
    .replace(team_name_mapping, subset=["homeTeamName", "opponentTeamName"])
    .sort('date')
)

df_pl_match_stats_22_23_filtrado_mapped.show()

+----------+--------------------+--------------------+----+----+---+---+---+---+---+-----+-----+-----+
|      date|        homeTeamName|    opponentTeamName|FTHG|FTAG|FTR| HS| AS|HST|AST| AvgH| AvgA| AvgD|
+----------+--------------------+--------------------+----+----+---+---+---+---+---+-----+-----+-----+
|2022-08-05|      Crystal Palace|             Arsenal|   0|   2|  A| 10| 10|  2|  2| 4.39| 1.88| 3.59|
|2022-08-06|              Fulham|           Liverpool|   2|   2|  D|  9| 11|  3|  4|10.99| 1.28| 6.05|
|2022-08-06|     AFC Bournemouth|         Aston Villa|   2|   0|  H|  7| 15|  3|  2|  3.8| 2.04|  3.5|
|2022-08-06|        Leeds United|Wolverhampton Wan...|   2|   1|  H| 12| 15|  4|  6| 2.34| 3.18| 3.34|
|2022-08-06|    Newcastle United|   Nottingham Forest|   2|   0|  H| 23|  5| 10|  0| 1.67| 5.57|  3.8|
|2022-08-06|   Tottenham Hotspur|         Southampton|   4|   1|  H| 18| 10|  8|  2| 1.36| 8.64| 5.27|
|2022-08-06|             Everton|             Chelsea|   0|   1|  A|  8| 

In [52]:
# ============================================================
# Validação de completude do join com as estatísticas de partida
# ============================================================
# Confere, no nível de jogo (não de evento), se toda partida presente no
# threat_dataset encontrou uma linha correspondente na base de odds/stats
# (join em date + homeTeamName + opponentTeamName).

df_games_threat = (
    df_threat
    .select('gameId', 'date', 'homeTeamName', 'opponentTeamName')
    .distinct()
)

df_games_join_check = join_match_stats(
    df_games_threat,
    df_pl_match_stats_22_23_filtrado_mapped
)

total_games = df_games_threat.count()
unmatched_games = df_games_join_check.filter(F.col('FTHG').isNull())
unmatched_count = unmatched_games.count()

print(f'Jogos no threat_dataset: {total_games}')
print(f'Jogos sem correspondência na base de estatísticas: {unmatched_count}')

unmatched_games.select('gameId', 'date', 'homeTeamName', 'opponentTeamName').show(50, truncate=False)

Jogos no threat_dataset: 361
Jogos sem correspondência na base de estatísticas: 0
+------+----+------------+----------------+
|gameId|date|homeTeamName|opponentTeamName|
+------+----+------------+----------------+
+------+----+------------+----------------+



### 2. Criação das bases agregadas e teste de correlação

### 2.1. Por Time-Partida

In [53]:
group_cols = [
    "gameId",
    "competitionName",
    "season",
    "date",
    "homeTeamName",
    "opponentTeamName",
    "homeTeam"
]

# zonas do campo consideradas nas colunas geradas no target_engineering
zone_suffixes = ['', '_half', '_third_2', '_third_3']

# threat_score_impact no lugar de threat_score — as 3 componentes normalizadas
# (progression_dist/total_players/atk_def_advantage) continuam as mesmas
agg_cols = []
for suffix in zone_suffixes:
    agg_cols += [
        f"threat_score_impact{suffix}",
        f"progression_dist{suffix}_norm",
        f"total_players{suffix}_norm",
        f"atk_def_advantage{suffix}_norm"
    ]

# sum no lugar de mean: soma é a agregação definida pro target do modelo
# (threat_score_impact é o efeito acumulado na posse, não uma média por
# evento). build_aggregated_df aplica UMA função pra toda a lista de
# agg_cols — então as 3 componentes normalizadas também são somadas aqui,
# não porque faça sentido pra elas isoladamente, mas porque essa é a
# limitação de manter a função genérica igual. Se quiser mean pra essas
# covariáveis e sum só pro target, precisa separar as duas listas e fazer
# 2 chamadas + join.
sum_cols, df_agg = build_aggregated_df(
    df=df_threat,
    group_cols=group_cols,
    agg_cols=agg_cols,
    agg_func=F.sum,
    agg_prefix="sum"
)

df_agg = join_match_stats(
    df_agg,
    df_pl_match_stats_22_23_filtrado_mapped
)

# build_defending_side_df no lugar de build_side_df: atribui a métrica ao
# time que DEFENDEU (não a quem atacava) e usa as estatísticas SOFRIDAS
# (do adversário) — ver docstring da função pra detalhes de por que é
# espelhado
df_team_match = (
    build_defending_side_df(df_agg, True, sum_cols)
    .unionByName(
        build_defending_side_df(df_agg, False, sum_cols)
    )
)

corr_cols = (
    df_team_match
    .drop(*(group_cols + ["teamName"]))
    .columns
)

# adf_team_match.show(5)
# plot_correlation_heatmap(df_team_match, corr_cols)
# plot_pvalue_heatmap(df_team_match, corr_cols)

### 2.2. Por Time-Partida-Ciclo de Posse

In [ ]:
# Configuração das agregações por grupo de coluna (fácil de trocar depois):
# uma função pra dentro do ciclo (CYCLE_AGG) e outra pra entre ciclos do
# time na partida (TEAM_AGG), separadas por "target" (threat_score_impact)
# e "components" (progression_dist/total_players/atk_def_advantage_norm).
# Ponto de partida: target = média da SOMA de cada ciclo (efeito acumulado
# do ciclo, depois a média entre os ciclos); components = média do MÁXIMO
# de cada ciclo, igual ao threat_score original (não faz sentido somar
# vantagem numérica/contagem de jogadores ao longo do ciclo).
CYCLE_AGG = {
    "target": F.sum,
    "components": F.max,
}
TEAM_AGG = {
    "target": F.mean,
    "components": F.mean,
}

possession_group_cols = [
    "gameId",
    "competitionName",
    "season",
    "date",
    "homeTeamName",
    "opponentTeamName",
    "homeTeam",
    "possession_id"
]

team_group_cols = [
    "gameId",
    "competitionName",
    "season",
    "date",
    "homeTeamName",
    "opponentTeamName",
    "homeTeam"
]

target_cols = [f"threat_score_impact{suffix}" for suffix in zone_suffixes]
component_cols = [c for c in agg_cols if c not in target_cols]

# 1ª etapa (dentro do ciclo): target somado, components pelo máximo —
# duas chamadas de build_aggregated_df (uma função só por chamada) + join
# pra juntar as duas de volta no nível de ciclo
sum_target_in_cycle_cols, df_possession_target = build_aggregated_df(
    df=df_threat,
    group_cols=possession_group_cols,
    agg_cols=target_cols,
    agg_func=CYCLE_AGG["target"],
    agg_prefix="sum"
)

max_components_in_cycle_cols, df_possession_components = build_aggregated_df(
    df=df_threat,
    group_cols=possession_group_cols,
    agg_cols=component_cols,
    agg_func=CYCLE_AGG["components"],
    agg_prefix="max"
)

df_possession = df_possession_target.join(df_possession_components, on=possession_group_cols, how="inner")

# 2ª etapa (entre ciclos do time na partida): média pros dois grupos —
# "avg_sum_threat_score_impact{suffix}" (média da soma por ciclo) e
# "avg_max_progression_dist{suffix}_norm" etc. (média do máximo por ciclo,
# igual ao threat_score original)
avg_target_cols, df_agg_target = build_aggregated_df(
    df=df_possession,
    group_cols=team_group_cols,
    agg_cols=sum_target_in_cycle_cols,
    agg_func=TEAM_AGG["target"],
    agg_prefix="avg"
)

avg_component_cols, df_agg_components = build_aggregated_df(
    df=df_possession,
    group_cols=team_group_cols,
    agg_cols=max_components_in_cycle_cols,
    agg_func=TEAM_AGG["components"],
    agg_prefix="avg"
)

df_agg = df_agg_target.join(df_agg_components, on=team_group_cols, how="inner")

agg_result_cols = avg_target_cols + avg_component_cols

df_agg = join_match_stats(
    df_agg,
    df_pl_match_stats_22_23_filtrado_mapped
)

# build_defending_side_df no lugar de build_side_df — mesma lógica invertida
# da seção 2.1 (ver docstring da função)
df_team_match = (
    build_defending_side_df(df_agg, True, agg_result_cols)
    .unionByName(
        build_defending_side_df(df_agg, False, agg_result_cols)
    )
)

corr_cols = (
    df_team_match
    .drop(*(group_cols + ["teamName"]))
    .columns
)

df_team_match.show(5)

plot_correlation_heatmap(df_team_match, corr_cols)
plot_pvalue_heatmap(df_team_match, corr_cols)

+---------------+---------+------+----------+-----------------+---------------------------+--------------------------------+-----------------------------------+-----------------------------------+-----------------------------+--------------------------+------------------------------+----------------------------------+-------------------------------+-----------------------------------+-------------------------------------+----------------------------------+--------------------------------------+-------------------------------------+----------------------------------+--------------------------------------+-----+-----+-----+------------+------------+
|competitionName|   season|gameId|      date|         teamName|avg_sum_threat_score_impact|avg_sum_threat_score_impact_half|avg_sum_threat_score_impact_third_2|avg_sum_threat_score_impact_third_3|avg_max_progression_dist_norm|avg_max_total_players_norm|avg_max_atk_def_advantage_norm|avg_max_progression_dist_half_norm|avg_max_total_players_hal

: 